In [ ]:
import sys
import os
from dotenv import load_dotenv

# Adds the parent directory (project root) to the python path
sys.path.append(os.path.abspath(os.path.join("../..")))
load_dotenv()

In [ ]:
import pandas as pd

documents_df = pd.read_parquet('../../data/raw/data/documents.parquet')

In [ ]:
# Configuration
BATCH_SIZE = 100000
OUTPUT_DIR = "../../data/processed/chunks/c_rec"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import pandas as pd
from dataclasses import asdict
from tqdm.auto import tqdm
from src.chunking_strategies.recursive_character_splitter import chunk_with_langchain

batch_chunks = []
batch_counter = 0

for row in tqdm(documents_df.itertuples(index=False), total=len(documents_df), desc="Streaming Chunks to Disk"):
    row_chunks = chunk_with_langchain(text=str(row.content), doc_id=str(row.doc_id))
    
    batch_chunks.extend([asdict(c) for c in row_chunks])
    
    # When the batch is full, dump it to disk and free RAM
    if len(batch_chunks) >= BATCH_SIZE * 5: 
        batch_df = pd.DataFrame(batch_chunks)
        batch_df.to_parquet(f"{OUTPUT_DIR}/batch_{batch_counter}.parquet", compression="snappy")
        
        batch_chunks.clear()
        batch_counter += 1

if batch_chunks:
    batch_df = pd.DataFrame(batch_chunks)
    batch_df.to_parquet(f"{OUTPUT_DIR}/batch_{batch_counter}.parquet", compression="snappy")
    batch_chunks.clear()

print(f"Done! All chunks safely saved as compressed files in '{OUTPUT_DIR}/'")



In [ ]:
import glob
import chromadb
import pandas as pd
import torch
from tqdm.auto import tqdm
from typing import cast
from chromadb.api.types import Metadatas, Documents, Embeddings, Embeddable
from chromadb.utils.embedding_functions import EmbeddingFunction
from sentence_transformers import SentenceTransformer

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME    = "all-MiniLM-L6-v2"  
EMBED_BATCH   = 512                   # tune up if VRAM allows (e.g. 1024, 2048)
CHROMA_BATCH  = 5_000                 # ChromaDB upsert ceiling
DEVICE        = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# ── Load model once ───────────────────────────────────────────────────────────
model = SentenceTransformer(MODEL_NAME, device=DEVICE) # type: ignore

# Use fp16 on GPU for ~2x speed + half the VRAM
if DEVICE == "cuda":
    model = model.half()

class STEmbeddingFunction(EmbeddingFunction[Embeddable]):
    def __call__(self, input: Documents) -> Embeddings:
        vecs = model.encode(
            input,
            batch_size=EMBED_BATCH,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vecs.tolist()

chroma_client = chromadb.PersistentClient(path="../../data/processed/embeddings")
collection = chroma_client.get_or_create_collection(name="c_rec")

# ── Ingest ────────────────────────────────────────────────────────────────────
batch_files = glob.glob("../../data/processed/chunks/c_rec/*.parquet")

for file_path in tqdm(batch_files, desc="Files"):
    df = pd.read_parquet(file_path)

    for start in tqdm(range(0, len(df), CHROMA_BATCH), desc="  Upserting", leave=False):
        sub = df.iloc[start : start + CHROMA_BATCH]

        ids       = sub["chunk_id"].tolist()
        documents = sub["text"].tolist()
        metadatas = cast(
            Metadatas,
            sub[["doc_id", "start_char", "end_char", "strategy"]].to_dict(orient="records"),
        )

        # Pre-compute embeddings explicitly on GPU in large batches
        embeddings = model.encode(
            documents,
            batch_size=EMBED_BATCH,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).tolist()

        collection.upsert(
            ids=ids,
            documents=documents,
            embeddings=embeddings, 
            metadatas=metadatas,
        )

    del df

print("Done — all chunks indexed into ChromaDB!")

In [1]:
import chromadb

chroma_client = chromadb.PersistentClient(path="../../data/processed/embeddings")
collection = chroma_client.get_or_create_collection(name="c_rec")
total_items = collection.count()
print(f"Total documents in collection: {total_items}")

Total documents in collection: 66085
